# 18 · DMID reference-mask audit sheets (CPU, about 2 min)
Runtime: CPU. Reads the jittered DMID crops (read only) and writes overlay sheets to `results/p2_dmid_audit/`: all 291 retained crops, the 29 retained multi-component crops, and the 19 removed duplicate records next to their retained counterparts. No model output is shown (blind to model errors).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
P = '/content/drive/MyDrive/quantum-placement'
D = f'{P}/data/external/dmid_mass_256_jit/roi'; OUT = f'{P}/results/p2_dmid_audit'; os.makedirs(OUT, exist_ok=True)
meta = pd.read_csv(f'{D}/metadata.csv'); keep = pd.read_csv(f'{P}/results/p2_analysis_v2/dmid_dedup.csv')[['roi_id', 'keep']]
meta = meta.merge(keep, on='roi_id')
def panel(ax, rid, title):
    im = cv2.imread(f'{D}/images/{rid}.png', 0); mk = cv2.imread(f'{D}/masks/{rid}.png', 0)
    rgb = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB); cs, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(rgb, cs, -1, (255, 60, 60), 2); ax.imshow(rgb); ax.set_title(title, fontsize=6); ax.axis('off')
def sheet(ids, titles, name, ncol=8):
    n = len(ids); nrow = int(np.ceil(n / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(2 * ncol, 2.1 * nrow), squeeze=False)
    for a in axs.ravel(): a.axis('off')
    for a, r, t in zip(axs.ravel(), ids, titles): panel(a, r, t)
    plt.tight_layout(); fig.savefig(f'{OUT}/{name}.png', dpi=90); plt.close(fig)
k = meta[meta.keep == True]
mc = k[k.n_comp > 1]
sheet(list(mc.roi_id), [f'{r.roi_id} {r.cls}/{r.sev} n={r.n_comp}' for r in mc.itertuples()], 'multi_component_29')
rem = meta[meta.keep == False]; ids, tt = [], []
for r in rem.itertuples():
    twin = k[(k.img == r.img) & (k.bbox_h == r.bbox_h) & (k.bbox_w == r.bbox_w)].roi_id.tolist()
    ids += [twin[0] if twin else r.roi_id, r.roi_id]; tt += [f'kept {twin[0] if twin else "?"}', f'removed {r.roi_id} {r.cls}/{r.sev}']
sheet(ids, tt, 'duplicates_19_pairs', ncol=8)
for i in range(0, len(k), 48):
    part = k.iloc[i:i + 48]; sheet(list(part.roi_id), [f'{r.roi_id} {r.cls} n={r.n_comp}' for r in part.itertuples()], f'all_retained_{i // 48 + 1}')
print('written to', OUT, sorted(os.listdir(OUT)))
